# Predictive Analysis of Wine Quality and Preferences Using Machine Learning Techniques

**Mini Hackathon 1 — KLH | 24–29 August 2026**

### Team
- G Sai Sathwik — 2520030328
- Guide: Dr. Shaik Asif

> **Dataset note:** The bundled CSV is an offline, reproducible demonstration dataset aligned to the UCI Wine Quality schema and published sample/quality structure. The notebook also contains a loader for the official UCI dataset. For a final empirical submission, place the official `winequality-red.csv` and `winequality-white.csv` files in `data/` and rerun all cells.


## 1. Problem Statement
Human sensory evaluation of wine is valuable but subjective, time-consuming and difficult to scale. The objective is to use physicochemical measurements to predict wine quality categories and estimate the quality score, providing an objective decision-support layer for quality assurance.

## 2. Objectives
1. Perform systematic EDA.
2. Identify distributions, outliers and feature relationships.
3. Compare Logistic Regression, SVM, Random Forest and Gradient Boosting.
4. Evaluate with accuracy, precision, recall and macro-F1.
5. Build a lightweight prediction prototype.
6. Explain limitations and future improvements.

## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
DATA_DIR = Path("data")


## 4. Load Data
The preferred route is to use the official UCI files. If they are not available, the bundled demo CSV is loaded so the notebook remains executable offline.

In [ ]:
red_path = DATA_DIR / "winequality-red.csv"
white_path = DATA_DIR / "winequality-white.csv"

if red_path.exists() and white_path.exists():
    red = pd.read_csv(red_path, sep=";")
    white = pd.read_csv(white_path, sep=";")
    red["wine_type"] = "red"
    white["wine_type"] = "white"
    df = pd.concat([red, white], ignore_index=True)
    print("Loaded official UCI files.")
else:
    df = pd.read_csv(DATA_DIR / "wine_quality_hackathon_demo.csv")
    print("Loaded bundled demonstration dataset.")

df.head()


## 5. Data Dictionary
The UCI source contains 11 physicochemical input variables and the sensory `quality` output. `wine_type` is added when red and white datasets are combined.

In [ ]:
data_dictionary = pd.DataFrame({
    "Feature": ["fixed acidity","volatile acidity","citric acid","residual sugar","chlorides",
                "free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol","quality"],
    "Role": ["Input"]*11 + ["Target"],
    "Description": [
        "Tartaric-acid-related fixed acidity",
        "Acetic-acid-related volatile acidity",
        "Citric acid concentration",
        "Residual sugar concentration",
        "Salt/chloride concentration",
        "Free sulfur dioxide",
        "Total sulfur dioxide",
        "Wine density",
        "Acidity/basicity measure",
        "Sulphate concentration",
        "Alcohol percentage by volume",
        "Sensory quality score"
    ]
})
data_dictionary


## 6. Data Quality Checks

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(15))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNumeric summary:")
display(df.describe().T)


## 7. EDA — Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
df["quality"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Wine Quality Score Distribution")
ax.set_xlabel("Quality")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


### EDA interpretation
The target is imbalanced: medium/normal quality wines dominate, while very low and very high scores are rare. This is why **macro-F1** is reported alongside accuracy.

## 8. EDA — Wine Type and Key Relationships

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4.5))
df["wine_type"].value_counts().plot(kind="bar", ax=axes[0], title="Wine Type")
df.boxplot(column="alcohol", by="quality", ax=axes[1])
axes[1].set_title("Alcohol by Quality")
axes[1].get_figure().suptitle("")
plt.tight_layout()
plt.show()


## 9. Correlation Analysis

In [ ]:
num_cols = [c for c in df.columns if c in [
    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides",
    "free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol","quality"
]]
corr = df[num_cols].corr()
plt.figure(figsize=(10,7))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=70, ha="right", fontsize=8)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=8)
plt.title("Correlation Matrix")
plt.colorbar()
plt.tight_layout()
plt.show()

print("Correlation with quality:")
display(corr["quality"].sort_values(ascending=False))


## 10. Feature Engineering
We convert the numeric quality score into three ordered classes:
- **Poor:** 3–4
- **Average:** 5–6
- **Excellent:** 7–10

This makes the primary task a multiclass classification problem while retaining the original numeric score for the optional regression task.

In [ ]:
df["quality_class"] = pd.cut(
    df["quality"], bins=[2,4,6,10],
    labels=["Poor","Average","Excellent"]
)
df["quality_class"].value_counts()


## 11. Model Training and Evaluation

In [ ]:
features = [
    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides",
    "free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol"
]
X = pd.get_dummies(df[features + ["wine_type"]], columns=["wine_type"])
y = df["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=3, class_weight="balanced"))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, min_samples_leaf=8,
        class_weight="balanced", max_features="sqrt",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        random_state=RANDOM_STATE
    )
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test,pred),
        "Precision (macro)": precision_score(y_test,pred,average="macro",zero_division=0),
        "Recall (macro)": recall_score(y_test,pred,average="macro",zero_division=0),
        "F1 (macro)": f1_score(y_test,pred,average="macro",zero_division=0)
    })

results = pd.DataFrame(rows).sort_values("F1 (macro)", ascending=False)
results


## 12. Best Model and Confusion Matrix

In [ ]:
best_name = results.iloc[0]["Model"]
best_model = models[best_name]
best_pred = best_model.predict(X_test)

print("Best model:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, labels=["Poor","Average","Excellent"]
)
plt.title(f"Confusion Matrix — {best_name}")
plt.show()


## 13. Feature Importance

In [ ]:
rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(10).to_frame("importance"))

importance.head(10).sort_values().plot(kind="barh", figsize=(8,5))
plt.title("Top Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 14. Research Connection

**Cortez et al. (2009)** introduced a wine-preference modeling approach using physicochemical tests on red and white Vinho Verde samples. Their work used regression and reported promising SVM performance, directly motivating our use of SVM and the same general feature family.

A later **Bhardwaj et al. (2022)** study demonstrates the use of synthetic data, feature selection and ensemble models for wine quality prediction, supporting the idea of using carefully controlled synthetic demonstrations when original experimental data are limited.

**Important:** our bundled offline CSV is a reproducible demonstration dataset. It is not claimed to be the original UCI observations.


## 15. Conclusion
The workflow demonstrates how routine physicochemical measurements can be transformed into a machine-learning decision-support system. The main challenge is target imbalance and overlap between neighboring quality categories. A practical system should therefore report macro-F1, calibration, feature importance and uncertainty rather than relying on accuracy alone.

### Official UCI replacement
For final validation, download the official `winequality-red.csv` and `winequality-white.csv` from the UCI Wine Quality repository, place them in `data/`, rerun the notebook, and update the PPT metrics with those real-data results.